# AITA Fault Detection - DistilRoBERTa Fine-tuning

This notebook contains the complete pipeline for: 
1. Data Preparation 
2. Baseline Training (Majority Class & TF-IDF)
3. DistilRoBERTa Fine-tuning on GPU
4. **Inference & Verdict Generation**

In [ ]:
import os

def ensure_data():
    if not os.path.exists('aita_verdicts_unique_6000.csv'):
        print("❌ 'aita_verdicts_unique_6000.csv' not found in session storage.")
        try:
            from google.colab import files
            print("Attempting to open upload widget...")
            uploaded = files.upload()
        except Exception as e:
            print(f"Could not open widget: {e}")
            print("PLEASE UPLOAD MANUALLY via the left sidebar.")
    else:
        print("'aita_verdicts_unique_6000.csv' is present and ready!")

ensure_data()

In [ ]:
!pip install transformers[torch] datasets accelerate scikit-learn pandas joblib tqdm

## 1. Data Preparation

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
import os

def prepare_data(input_file, output_dir):
    df = pd.read_csv(input_file)
    df['title'] = df['title'].fillna('')
    df['full post'] = df['full post'].fillna('')
    df['text'] = df['title'] + " " + df['full post']
    
    label_map = {'user_is_fault': 1, 'user_ok': 0}
    df['label'] = df['verdict'].map(label_map)
    df = df[['text', 'label', 'pid']]
    
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    train_df = train_df.reset_index(drop=True)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_df['fold'] = -1
    for fold, (t_idx, v_idx) in enumerate(skf.split(train_df['text'], train_df['label'])):
        train_df.loc[v_idx, 'fold'] = fold
        
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    train_df.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
    test_df.to_csv(os.path.join(output_dir, 'test.csv'), index=False)
    return train_df, test_df

if os.path.exists('aita_verdicts_unique_6000.csv'):
    train_df, test_df = prepare_data('aita_verdicts_unique_6000.csv', 'data_splits')
    print("Data Preparation Complete.")
else:
    print("Stop! Please upload the file first.")

Data Preparation Complete.


## 2. Baselines

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

if 'train_df' in locals():
    X_train, y_train = train_df['text'], train_df['label']
    X_test, y_test = test_df['text'], test_df['label']

    print("--- Majority Baseline ---")
    maj = y_train.mode()[0]
    print(f"Test Accuracy: {accuracy_score(y_test, [maj]*len(y_test)):.4f}")

    print("\n--- TF-IDF + LR ---")
    v = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
    X_tr_tf = v.fit_transform(X_train)
    X_ts_tf = v.transform(X_test)
    clf = LogisticRegression(max_iter=1000).fit(X_tr_tf, y_train)
    print(f"Test Accuracy: {accuracy_score(y_test, clf.predict(X_ts_tf)):.4f}")
else:
    print("train_df not found. Skipping baselines.")

--- Majority Baseline ---
Test Accuracy: 0.5141

--- TF-IDF + LR ---
Test Accuracy: 0.6859


## 3. DistilRoBERTa Fine-tuning

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset

if 'train_df' in locals():
    model_name = "distilroberta-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch): return tokenizer(batch["text"], truncation=True, padding=True, max_length=256)

    ds_train = Dataset.from_pandas(train_df).map(tokenize, batched=True)
    ds_test = Dataset.from_pandas(test_df).map(tokenize, batched=True)

    train_split = ds_train.filter(lambda x: x['fold'] != 0)
    val_split = ds_train.filter(lambda x: x['fold'] == 0)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=-1)
        return {"accuracy": accuracy_score(p.label_ids, preds), "f1": f1_score(p.label_ids, preds, average='macro')}

    args = TrainingArguments(
        output_dir="results",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none"
    )

    trainer = Trainer(
        model=model, args=args, train_dataset=train_split, eval_dataset=val_split,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer), compute_metrics=compute_metrics
    )

    trainer.train()

    print("\n--- Transformer Hold-out Test ---")
    res = trainer.predict(ds_test)
    print(f"Accuracy: {accuracy_score(res.label_ids, np.argmax(res.predictions, axis=-1)):.4f}")
else:
    print("train_df not found. Skipping transformer training.")

## 4. Inference (Verdict Generation)

Use this section to test the model on your own posts or batch process new data.

In [ ]:
def get_verdict(title, post_content):
    text = f"{title} {post_content}"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(model.device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    pred = torch.argmax(logits, dim=-1).item()
    label = "USER_IS_FAULT (Asshole)" if pred == 1 else "USER_OK (Not the Asshole)"
    return label

# Example Test
test_title = "AITA for not doing the dishes when it was my turn?"
test_body = "I had a really long day at work and my roommate got mad because I left the plates in the sink. I usually do them but I was just exhausted."
print(f"Verdict: {get_verdict(test_title, test_body)}")

### Process `posts_sample_dataset.csv` (Batch Inference)
If you upload `posts_sample_dataset.csv`, this code will create a new file with verdicts.

In [ ]:
if os.path.exists('posts_sample_dataset.csv'):
    sample_df = pd.read_csv('posts_sample_dataset.csv')
    print(f"Processing {len(sample_df)} sample posts...")
    
    sample_df['ai_verdict'] = sample_df.apply(
        lambda row: get_verdict(row['title'], row['full post'] if 'full post' in row else row['post']),
        axis=1
    )
    
    sample_df.to_csv('sample_with_ai_verdicts.csv', index=False)
    print("Saved results to 'sample_with_ai_verdicts.csv'!")
else:
    print("Note: 'posts_sample_dataset.csv' not found in session storage. Skip batch inference.")